<a href="https://colab.research.google.com/github/muhammadabdurrehmanmaqsood/flyrank-ml-internship-abdurrehman/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadabdurrehmanmaqsood/flyrank-ml-internship-abdurrehman/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The Rule:** A piece of content requires a refresh if it is getting stale (over 180 days since the last update) and its average position is slipping outside the top 10, provided we actually have position data for it.

**Reason Code:** `stale_and_slipping`

**Signal Check 1: Staleness (FlyRank Flag)**
Hypothesis: Older content (days_since_update > 180) is more likely to have a declining traffic trend.
Verdict: **CONFIRMED**

**Signal Check 2: Position Data Presence**
Hypothesis: Items with a valid average position (not 0) behave differently in engagement than items missing position data.
Verdict: **MIXED**

In [3]:
import os

# 1. Cloning specific repository into the Colab environment
if not os.path.exists('flyrank-ml-internship-abdurrehman'):
    !git clone https://github.com/muhammadabdurrehmanmaqsood/flyrank-ml-internship-abdurrehman.git

# 2. Change the current working directory to where the notebook expects to be
# This makes all the '../../' relative paths work perfectly
os.chdir('flyrank-ml-internship-abdurrehman/work/notebooks')

print("Repository cloned and working directory set to:", os.getcwd())

Cloning into 'flyrank-ml-internship-abdurrehman'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 143 (delta 51), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.86 MiB | 11.13 MiB/s, done.
Resolving deltas: 100% (51/51), done.
Repository cloned and working directory set to: /content/flyrank-ml-internship-abdurrehman/work/notebooks/flyrank-ml-internship-abdurrehman/work/notebooks


In [6]:
# Print all column names to find the correct one
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [8]:
print("--- Signal Check 1: Staleness ---")
# Using the column days_since_last_update
df['is_stale_bucket'] = np.where(df['days_since_last_update'] >= 180, 'Stale (>=180d)', 'Fresh (<180d)')
staleness_table = df.groupby('is_stale_bucket').agg(
    n=('content_id', 'count'),
    avg_engagement_rate=('engagement_rate', 'mean')
).reset_index()
print(staleness_table)
print("\nVerdict: CONFIRMED - We observe a directional shift in engagement based on staleness.\n")

print("--- Signal Check 2: Position Data Presence ---")
# avg_position = 0 means no data, not rank zero.
df['has_position_data'] = np.where(df['avg_position'] == 0, 'No Data (0)', 'Has Data (>0)')
position_table = df.groupby('has_position_data').agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean')
).reset_index()
print(position_table)
print("\nVerdict: MIXED - Presence of data changes metrics, but requires filtering by content_type to be purely predictive.")

--- Signal Check 1: Staleness ---
  is_stale_bucket      n  avg_engagement_rate
0   Fresh (<180d)  29826             2.537473
1  Stale (>=180d)    174             2.028333

Verdict: CONFIRMED - We observe a directional shift in engagement based on staleness.

--- Signal Check 2: Position Data Presence ---
  has_position_data      n   avg_ctr
0     Has Data (>0)  28795  0.519662
1       No Data (0)   1205  0.297369

Verdict: MIXED - Presence of data changes metrics, but requires filtering by content_type to be purely predictive.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Encoding the score using unweighted, human-readable logic. We isolate items with valid position data, apply our staleness threshold, and rank them by their overall visibility (impressions) to prioritize high-impact refreshes.

In [10]:
# 1. Base conditions
condition_stale = (df['days_since_last_update'] >= 180).astype(int)
condition_slipping = (df['avg_position'] > 10).astype(int)
condition_valid_pos = (df['avg_position'] > 0).astype(int)

# 2. Score calculation
df['baseline_score'] = condition_stale * condition_slipping * condition_valid_pos * df['impressions_90d']

# 3. Attach reason code and action label
df['reason_code'] = np.where(df['baseline_score'] > 0, 'stale_and_slipping', 'no_action_needed')
df['action_label'] = np.where(df['baseline_score'] > 0, 'REFRESH', 'HOLD')

# 4. Rank the queue (highest score first)
ranked_queue = df[df['baseline_score'] > 0].sort_values(by='baseline_score', ascending=False).copy()

import os

# 5. Output to CSV
output_dir = '../../work/outputs'
# This creates the directory safely if it doesn't exist yet
os.makedirs(output_dir, exist_ok=True)

output_columns = ['content_id', 'client_id', 'baseline_score', 'reason_code', 'action_label']
ranked_queue[output_columns].to_csv(f'{output_dir}/baseline_action_score.csv', index=False)

print(f"Ranked queue generated with {len(ranked_queue)} actionable items.")
print("Saved to: work/outputs/baseline_action_score.csv")

Ranked queue generated with 63 actionable items.
Saved to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
# Display the top 20 for manual review
top_20 = ranked_queue[['content_id', 'days_since_last_update', 'avg_position', 'impressions_90d', 'reason_code', 'action_label']].head(20)
display(top_20)

,content_id,days_since_last_update,avg_position,impressions_90d,reason_code,action_label
16751,content_cf56e2e2e282,194,19.7,61678,stale_and_slipping,REFRESH
16514,content_7368877ea310,194,24.8,59472,stale_and_slipping,REFRESH
7021,content_1bfaa38ff26c,194,22.2,25715,stale_and_slipping,REFRESH
21268,content_0a91db491d14,193,10.5,13299,stale_and_slipping,REFRESH
11489,content_5feee3994adb,194,39.0,7812,stale_and_slipping,REFRESH
12045,content_c2d929d83eaa,193,17.9,7558,stale_and_slipping,REFRESH
698,content_b16bd7307b39,194,31.0,4590,stale_and_slipping,REFRESH
5327,content_fe16a55cd13d,194,16.4,4556,stale_and_slipping,REFRESH
26810,content_ecb6215e79fd,194,25.3,4429,stale_and_slipping,REFRESH
20837,content_928af3e22c80,193,15.8,1697,stale_and_slipping,REFRESH


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

### Top-20 Manual Review & Weak Picks
Every item below has an action of **REFRESH** and a reason code of **stale_and_slipping** because they met our baseline thresholds (days_since_last_update >= 180, avg_position > 10, and impressions_90d > 0). Here is the critical assessment of when recommending these actions would result in a bad pick:

1. **content_cf56e2e2e282** (Avg Pos: 19.7, Impr: 61,678) — **Wrong if:** The high impression volume is driven by a single hyper-viral seasonal keyword that has already passed its lifecycle peak, meaning a refresh won't recapture traffic.
2. **content_7368877ea310** (Avg Pos: 24.8, Impr: 59,472) — **Wrong if:** The page ranks deep for a high-volume generic head term dominated by giant authoritative domains, where a text refresh alone cannot bridge the domain authority gap.
3. **content_1baa38ff26cc** (Avg Pos: 22.2, Impr: 25,715) — **Wrong if:** This page belongs to a discontinued product line or service that the business no longer converts or supports, rendering a update useless for ROI.
4. **content_0a91db491d14** (Avg Pos: 10.5, Impr: 13,299) — **Wrong if:** The page is right on the cusp of page 1 (10.5) and is experiencing a temporary fluctuation; an immediate major rewrite risks disrupting the keywords it *is* successfully stabilizing for.
5. **content_5feee3994adb** (Avg Pos: 39.0, Impr: 7,812) — **Wrong if:** An average position of 39.0 means it is buried on page 4; a basic refresh is a weak pick because it likely needs a complete intent pivot or structural URL consolidation rather than a simple update.
6. **content_c2d929d83eaa** (Avg Pos: 17.9, Impr: 7,558) — **Wrong if:** The page contains evergreen reference data or static compliance documents where updating the timestamp text adds no value to user intent.
7. **content_b1bd7307b399** (Avg Pos: 31.0, Impr: 4,590) — **Wrong if:** The impression footprint is bloated by tracking impressions from low-intent international locales where the client doesn't sell.
8. **content_fe16a55cd13d** (Avg Pos: 16.4, Impr: 4,556) — **Wrong if:** The ranking drop was caused by site-wide technical debt (e.g., slow core web vitals on this content type) which content edits cannot fix.
9. **content_ecb6215e79fd** (Avg Pos: 25.3, Impr: 4,429) — **Wrong if:** The article is an old news round-up or specific event announcement from over 190 days ago; refreshing historical news breaks chronological sense.
10. **content_928af3e22c80** (Avg Pos: 15.8, Impr: 1,697) — **Wrong if:** The search volume for this niche topic is contracting organically industry-wide, making a content refresh a waste of editorial resource.
11. **content_bdbec75c1148** (Avg Pos: 21.8, Impr: 1,316) — **Wrong if:** This URL is intentionally canonicalized or redirected to a newer hub page, meaning the metrics we see are just tail-end tracking anomalies.
12. **content_77d4d5930e5e** (Avg Pos: 18.6, Impr: 828) — **Wrong if:** The page targets long-tail user queries that are now being completely answered directly in the search engine's AI overviews, destroying organic CTR regardless of content quality.
13. **content_6226ee6adc91** (Avg Pos: 17.8, Impr: 545) — **Wrong if:** The page belongs to a B2B client whose target audience values deep, static whitepapers over frequent, superficial modifications.
14. **content_074ba6ead17b** (Avg Pos: 48.0, Impr: 533) — **Wrong if:** Position 48 implies it's functionally invisible to users; expending effort here yields virtually zero traffic return compared to optimizing page-2 items.
15. **content_b65fe2792b44** (Avg Pos: 16.7, Impr: 371) — **Wrong if:** The content is a highly tailored landing page built strictly for a paid ad campaign (PPC) and was never meant to rank highly in organic search.
16. **content_6476d1d8c050** (Avg Pos: 67.8, Impr: 304) — **Wrong if:** At 313 days old and rank 67.8, this page is thoroughly dead; it should be deleted or completely merged into a broader pillar page rather than refreshed.
17. **content_4f241bad48a3** (Avg Pos: 19.1, Impr: 285) — **Wrong if:** The user intent for this keyword has structurally shifted from text-based articles to video/tools, rendering our text-based layout obsolete.
18. **content_d25a099b3726** (Avg Pos: 64.5, Impr: 202) — **Wrong if:** The low impression volume indicates a lack of market interest, creating a poor ROI priority choice for the content operations team.
19. **content_e444c00065bd** (Avg Pos: 16.6, Impr: 148) — **Wrong if:** This page was generated as part of a localized programmatic SEO test that the company decided to deprecate.
20. **content_b6e4581523ed** (Avg Pos: 14.0, Impr: 104) — **Wrong if:** The page yields only ~1 impression a day; prioritizing it in the top 20 queue over pages with 60,000+ impressions is a major baseline structural logic flaw.

### Emerging Patterns in Failure Modes
Looking over the entire top 20, a distinct structural trend stands out:
* **The Low-ROI Tail:** In the bottom half of the list (items 15-20), our rule prioritizes pages with fewer than 500 total impressions over 90 days simply because they match the arbitrary "stale and slipping" criteria.
* **The Page 4+ Gravitational Pull:** Items like 5, 14, and 16 have average positions deeper than 35+. A standard content refresh is mathematically highly unlikely to salvage their organic ranking without deep structural site changes.
* **Correction for Next Week's ML Model:** Our rule lacks an explicit engagement/traffic floor. In the next iteration, we must incorporate a minimum impression or pageview threshold to filter out these low-value items from clogging up the high-priority queue.

### Leakage Check Confirmation
* **Future-Window Leakage:** Confirmed absent. All inputs (`days_since_last_update`, `avg_position`, `impressions_90d`) are calculated entirely within the historical trailing snapshot window.
* **Label-Derived Inputs:** Confirmed absent. We successfully avoided the label trap by omitting `trend_pct` and `trend_direction` entirely from our feature engineering and scoring cells.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.